# Practical No. 06: Retrieval-Augmented Generation (RAG) using a Pre-trained LLM

**Aim:** Build a RAG application that answers questions from a domain-specific document collection, then compare its answers with the same pre-trained LLM without retrieval.



In [1]:
# 1. Install required libraries
!pip -q install sentence-transformers transformers accelerate pypdf python-docx

In [2]:
# 2. Import libraries
import os
import re
import numpy as np
import torch

from google.colab import files
from pypdf import PdfReader
from docx import Document
from sentence_transformers import SentenceTransformer
from transformers import pipeline

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch: 2.14.1+cu130
GPU available: True


In [3]:
# 3. Upload your domain-specific documents
# Supported formats: PDF, TXT, DOCX
uploaded = files.upload()

def extract_text(file_name):
    ext = os.path.splitext(file_name)[1].lower()

    if ext == ".pdf":
        reader = PdfReader(file_name)
        return "\\n".join(page.extract_text() or "" for page in reader.pages)

    elif ext == ".txt":
        with open(file_name, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()

    elif ext == ".docx":
        doc = Document(file_name)
        return "\\n".join(p.text for p in doc.paragraphs)

    else:
        print(f"Skipping unsupported file: {file_name}")
        return ""

documents = []
for file_name in uploaded.keys():
    text = extract_text(file_name)
    if text.strip():
        documents.append({"source": file_name, "text": text})

if not documents:
    raise ValueError("No readable text found. Upload a text-based PDF, TXT, or DOCX file.")

print(f"Loaded {len(documents)} document(s).")
for doc in documents:
    print(f"- {doc['source']}: {len(doc['text'])} characters")

Saving NFA 10.pdf to NFA 10 (1).pdf
Loaded 1 document(s).
- NFA 10 (1).pdf: 4082 characters


In [4]:
# 4. Split documents into overlapping chunks
def chunk_text(text, chunk_size=800, overlap=150):
    # Normalize excessive whitespace while preserving readable sentence boundaries
    text = re.sub(r"\\s+", " ", text).strip()
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == len(text):
            break
        start = end - overlap

    return chunks

chunks = []
for doc in documents:
    for i, chunk in enumerate(chunk_text(doc["text"])):
        chunks.append({
            "text": chunk,
            "source": doc["source"],
            "chunk_id": i
        })

print("Total chunks created:", len(chunks))
print("\\nExample chunk:\\n", chunks[0]["text"][:500])

Total chunks created: 7
\nExample chunk:\n Yon detekminiic Fnite Atontata ( ¥FA) 
The FA hich has o lo) (A) One (t) (d) 
wo avsitionh t ainpl ualood jep a ale 
i Calleol a NFA 
in nulleple stats at he bane ANFA Carn Resicle 
ime 
NFA h 5 tple in 2, d, , F , F 
-Set a all tats b 
/P alphcbes 
Qxz 2na deauition un 
-ivi tHl &tte 
F Sot all inal Stuts 
autonata iin 1 
ztake La tot pu 
a ct coil be 
(o) , 
|,4, 
- 
o,4.1, 111,11, 
-11, F \nWe abb 
b 
AcCeptd 
Note 
NFA iA Aough idLa u ptant aagsl Aaungc 
ACcephng pouwek NEA 
= ACdephn pouset


In [5]:
# 5. Create embeddings and build the retrieval index
# Embeddings convert text into vectors. Similarity search retrieves relevant chunks.
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

chunk_texts = [item["text"] for item in chunks]
document_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding matrix shape:", document_embeddings.shape)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape: (7, 384)


In [6]:

import transformers
print(transformers.__version__)


4.57.3


In [7]:

# 6. Load the pre-trained LLM

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

generator = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

print("LLM loaded successfully!")


Device set to use cuda:0


LLM loaded successfully!


In [9]:

# 7. Define retrieval and answer-generation functions

def retrieve(query, top_k=3):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    # Calculate cosine similarity
    scores = document_embeddings @ query_embedding
    top_indices = np.argsort(scores)[::-1][:min(top_k, len(chunks))]

    results = []
    for idx in top_indices:
        item = chunks[int(idx)].copy()
        item["score"] = float(scores[idx])
        results.append(item)

    return results


def generate_answer(prompt, max_new_tokens=180):
    result = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        truncation=True
    )
    return result[0]["generated_text"].strip()


def answer_with_rag(question, top_k=3):
    retrieved = retrieve(question, top_k=top_k)

    context = "\n\n".join(
        f"[Source: {item['source']}] {item['text']}"
        for item in retrieved
    )

    prompt = f"""Answer the question using only the context below.
If the answer is not contained in the context, say:
"I could not find this information in the provided documents."

Context:
{context}

Question: {question}
Answer:"""

    answer = generate_answer(prompt)
    return answer, retrieved


def answer_without_rag(question):
    prompt = f"""Answer the following question using your general knowledge.
If you do not know, say that you are not sure.

Question: {question}
Answer:"""

    return generate_answer(prompt)


print("Retrieval and generation functions defined successfully!")


Retrieval and generation functions defined successfully!


In [13]:
# 8. Ask a question and compare both approaches
question = "What is a Non-Deterministic Finite Automaton (NFA)? Explain its five components."

rag_answer, retrieved_chunks = answer_with_rag(question)
baseline_answer = answer_without_rag(question)

print("=" * 80)
print("QUESTION:", question)
print("\n--- RAG ANSWER ---")
print(rag_answer)

print("\n--- LLM WITHOUT RETRIEVAL ---")
print(baseline_answer)

print("\n--- RETRIEVED DOCUMENT CHUNKS ---")
for rank, item in enumerate(retrieved_chunks, start=1):
    print(f"\n{rank}. Source: {item['source']} | Similarity: {item['score']:.3f}")
    print(item["text"][:700])

QUESTION: What is a Non-Deterministic Finite Automaton (NFA)? Explain its five components.

--- RAG ANSWER ---
NFA 10 (1).pdf] a) Note: cn uge DFA NFA No n+ n+2 1. W=Sx s| = n n + L xS , Is) = n n + W xSx , IS] = 5 2. Stoks 8 ends coithsame bol Sterk ends it db5" mbol nt2 t Sstnbo nt Spasbul om Riaht nt 2 . IO= n + 2 I n = (mocl n) n*Convession oh NFA to DFA 7he ptwtw NFA tr DFA Cog version is

--- LLM WITHOUT RETRIEVAL ---
Non-Deterministic Finite Automaton

--- RETRIEVED DOCUMENT CHUNKS ---

1. Source: NFA 10 (1).pdf | Similarity: 0.394
a) 
Note: 
cn uge DFA NFA No 
n+ n+2 
1. W=Sx s| = n 
n+ L xS , Is) = n 
n+ W xSx , IS] = 
5 
2. Stoks 8 ends 
coithsame 
bol 
Sterk ends it db5" 
mbol 
nt2 
t Sstnbo 
nt Spasbul om Riaht 
nt 2 
. 
IO= n 
+2 I n 
= (mocl n) \n*Convession oh NFA to DFA 
7he ptwtw NFA tr DFA Cog version 
is called as Aubset Convewion 
Aaitht t M-CA,z, ), L, F) 
-
NFA 
M-(a, c,J', F') -DFA 
Tnitial State : 
No lane 
in initiol slae 
Con&tention : 
S(,,z) = &(li.«) 
&'(t 

In [11]:
# 9. Interactive question-answering loop
while True:
    question = input("\\nEnter your question (or type 'exit' to stop): ").strip()
    if question.lower() in {"exit", "quit", "q"}:
        print("Session ended.")
        break
    if not question:
        print("Please enter a question.")
        continue

    rag_answer, retrieved_chunks = answer_with_rag(question)
    baseline_answer = answer_without_rag(question)

    print("\\n[RAG ANSWER]\\n", rag_answer)
    print("\\n[LLM WITHOUT RETRIEVAL]\\n", baseline_answer)
    print("\\n[RETRIEVED SOURCES]")
    for item in retrieved_chunks:
        print(f"- {item['source']} (similarity: {item['score']:.3f})")

\nEnter your question (or type 'exit' to stop): What is a Non-Deterministic Finite Automaton (NFA)? Explain its five components.
\n[RAG ANSWER]\n NFA 10 (1).pdf] a) Note: cn uge DFA NFA No n+ n+2 1. W=Sx s| = n n + L xS , Is) = n n + W xSx , IS] = 5 2. Stoks 8 ends coithsame bol Sterk ends it db5" mbol nt2 t Sstnbo nt Spasbul om Riaht nt 2 . IO= n + 2 I n = (mocl n) n*Convession oh NFA to DFA 7he ptwtw NFA tr DFA Cog version is
\n[LLM WITHOUT RETRIEVAL]\n Non-Deterministic Finite Automaton
\n[RETRIEVED SOURCES]
- NFA 10 (1).pdf (similarity: 0.394)
- NFA 10 (1).pdf (similarity: 0.393)
- NFA 10 (1).pdf (similarity: 0.374)
\nEnter your question (or type 'exit' to stop): Explain acceptance of strings by NFA.
\n[RAG ANSWER]\n (A) One (t) (d) wo avsitionh t ainpl ualood jep a ale i Calleol a NFA in nulleple stats at he bane ANFA Carn Resicle ime NFA h 5 tple in 2, d, , F , F -Set a all tats b /P alphcbes Qxz 2na deauition un -ivi tHl &tte F Sot all inal Stuts autonata iin 1 ztake La tot pu a


## 10. Result and observations

The RAG application was tested using questions from the uploaded NFA 10.pdf document.

| Evaluation point | RAG (retrieval + LLM) | LLM without retrieval |
|---|---|---|
| Information source | Retrieved chunks from NFA notes | Model's pre-trained knowledge |
| NFA concepts | Uses information from uploaded notes | Uses general knowledge of NFA |
| Grounding | Answers are conditioned on retrieved context | No document grounding |
| Main limitation | Retrieval may return irrelevant or incomplete chunks | May give generic or unsupported answers |

**Conclusion:** The RAG system retrieves relevant text chunks from the uploaded NFA document and supplies them as context to a pre-trained LLM. Compared with using the LLM alone, RAG can produce answers that are more relevant to the supplied document collection. Its performance still depends on document quality, chunking, retrieval relevance, and the LLM's ability to follow instructions.
